# ComfyUI Kaggle Image Editor

Run with GPU enabled. Attach a Kaggle Dataset containing a compatible Stable Diffusion 1.5 checkpoint. Kaggle sessions are temporary. Public Gradio/Cloudflare links can be accessed by anyone who has the URL; do not upload private images.

In [ ]:
# 1. Check GPU and import helpers
!nvidia-smi
import os, subprocess, time, json, pathlib, platform, urllib.request


In [ ]:
# 2. Clone this repository
REPO_URL = 'https://github.com/AnantPramanik/rouge.git'
PROJECT = '/kaggle/working/rouge'
if not os.path.isdir(PROJECT): subprocess.run(['git','clone',REPO_URL,PROJECT],check=True)
os.chdir(PROJECT)


In [ ]:
# 3. Install ComfyUI and Gradio dependencies
subprocess.run(['bash','scripts/install.sh'],check=True)
subprocess.run(['bash','scripts/install_nodes.sh'],check=True)
!pip install -r requirements.txt


In [ ]:
# 4. List model files in attached Kaggle datasets
for root, dirs, files in os.walk('/kaggle/input'):
    for name in files:
        if name.lower().endswith(('.safetensors','.ckpt','.pt')): print(os.path.join(root,name))


In [ ]:
# 5. Set the directory of your attached model dataset, then copy/link a checkpoint
MODEL_DATASET_DIR = ''  # Example: '/kaggle/input/my-sd15-model'
checkpoint_dir = '/kaggle/working/ComfyUI/models/checkpoints'
os.makedirs(checkpoint_dir,exist_ok=True)
if MODEL_DATASET_DIR:
    candidates=[os.path.join(r,f) for r,ds,fs in os.walk(MODEL_DATASET_DIR) for f in fs if f.lower().endswith(('.safetensors','.ckpt','.pt'))]
    if not candidates: raise RuntimeError('No checkpoint found in MODEL_DATASET_DIR')
    chosen=candidates[0]; dest=os.path.join(checkpoint_dir,os.path.basename(chosen))
    if not os.path.exists(dest): os.symlink(chosen,dest)
    print('Using',dest)
else: print('Set MODEL_DATASET_DIR before generation; no model copied.')


In [ ]:
# 6. Configure the workflow if a model exists
ckpts=[f for f in os.listdir(checkpoint_dir) if f.lower().endswith(('.safetensors','.ckpt','.pt'))]
workflow_path=os.path.join(PROJECT,'workflows','image_edit_api.json')
with open(workflow_path,encoding='utf-8') as f: wf=json.load(f)
if ckpts:
    wf['1']['inputs']['ckpt_name']=ckpts[0]
    with open(workflow_path,'w',encoding='utf-8') as f: json.dump(wf,f,indent=2)
    print('Workflow checkpoint:',ckpts[0])


In [ ]:
# 7. Start ComfyUI
comfy_log=open('/kaggle/working/comfyui.log','w')
comfy_process=subprocess.Popen(['python','main.py','--listen','0.0.0.0','--port','8188'],cwd='/kaggle/working/ComfyUI',stdout=comfy_log,stderr=subprocess.STDOUT)
time.sleep(15)
print('ComfyUI PID:',comfy_process.pid)


In [ ]:
# 8. Start the Gradio browser editor
env=os.environ.copy(); env['WORKFLOW_PATH']=workflow_path
gradio_log=open('/kaggle/working/gradio.log','w')
gradio_process=subprocess.Popen(['python',os.path.join(PROJECT,'app','gradio_app.py')],cwd=PROJECT,env=env,stdout=gradio_log,stderr=subprocess.STDOUT)
time.sleep(12)
print('Gradio PID:',gradio_process.pid,'; inspect /kaggle/working/gradio.log for its share URL')


In [ ]:
# 9. Optional Cloudflare quick tunnel for the native ComfyUI interface
arch=platform.machine().lower()
url='https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64' if arch in ('x86_64','amd64') else ('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-arm64' if arch in ('aarch64','arm64') else None)
if url:
    try:
        binary='/kaggle/working/cloudflared'; urllib.request.urlretrieve(url,binary); os.chmod(binary,0o755)
        tunnel_log=open('/kaggle/working/comfyui-tunnel.log','w')
        tunnel_process=subprocess.Popen([binary,'tunnel','--url','http://127.0.0.1:8188'],stdout=tunnel_log,stderr=subprocess.STDOUT)
        print('Tunnel launched; inspect comfyui-tunnel.log for the public URL.')
    except Exception as e: print('Tunnel unavailable:',repr(e))
else: print('Unsupported architecture')


In [ ]:
# 10. Inspect service logs
for p in ['/kaggle/working/comfyui.log','/kaggle/working/gradio.log','/kaggle/working/comfyui-tunnel.log']:
    print('\n====',p,'====')
    if os.path.exists(p): print(open(p,errors='ignore').read()[-5000:])
    else: print('Not created yet')


## Open the interfaces

- Gradio share URL in `gradio.log`: simple upload + prompt editor.
- `trycloudflare.com` URL in `comfyui-tunnel.log`: full ComfyUI node editor.
- Tunnels are best-effort and may be blocked by Kaggle/network policy. Save outputs before stopping the temporary session.